# Day 2 · scikit-image: counting nuclei

**Block:** scikit-image (45 min, including slides and live coding)

**Task:** count the nuclei in a fluorescence microscopy image of human cells, and
measure their sizes. You will:

1. **Look** at the image as a NumPy array (3 min)
2. **Filter** it to reduce noise (4 min)
3. **Threshold** it into foreground and background (8 min)
4. **Clean up and label** the objects, then count them (8 min)
5. **Measure** each object with `regionprops` (7 min)

Sections 1 to 3 are the first "Your turn" slide, sections 4 and 5 the second.
6. **Stretch:** separate touching nuclei with a watershed, then try the SPL coins
   exercises

The image is `skimage.data.human_mitosis()`: human cells in culture,
some of them dividing, stained for DNA. It comes from Moffat *et al.*, *Cell*
124:1283 (2006), courtesy of David Root, licensed CC0.

Further material: [SPL, scikit-image: image
processing](https://lectures.scientific-python.org/packages/scikit-image/index.html)
and [Image manipulation and processing using NumPy and
SciPy](https://lectures.scientific-python.org/advanced/image_processing/index.html)
(CC BY 4.0), plus the [BioImage Analysis
Notebooks](https://haesleinhuepf.github.io/BioImageAnalysisNotebooks/).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy as sp
import skimage as ski

## 1. Look at the image

Load `../data/human_mitosis.png` with `ski.io.imread`, then answer:

* What are its shape, dtype, minimum and maximum?
* Display it with `imshow` and a grey colormap. Then plot a histogram of the pixel
  values (`ax.hist(image.ravel(), bins=...)`). What do the two humps tell you?

In [ ]:
image = ski.io.imread("../data/human_mitosis.png")

In [ ]:
# Your code here


*Your answer here.*

## 2. Filter

Thresholding single pixels is sensitive to noise. Smooth the image with
`ski.filters.gaussian` using `sigma=1`, and call the result `smoothed`. Show
the image and `smoothed` side by side, zoomed in on a corner (for example
`[:100, :100]`).

What is the dtype and range of `smoothed`? Why did it change?

In [ ]:
# Your code here


*Your answer here.*

## 3. Threshold

1. Compute an automatic threshold for `smoothed` with `ski.filters.threshold_otsu`.
2. Make the boolean image `binary = smoothed > threshold` and display it.
3. Try `ski.filters.try_all_threshold(smoothed)`. Which methods look better or
   worse than Otsu here?

In [ ]:
# Your code here


In [ ]:
# Your code here


*Your answer here.*

## 4. Clean up, label and count

1. Remove specks smaller than about 30 pixels with
   `ski.morphology.remove_small_objects(binary, max_size=30)`.
2. Remove nuclei cut off by the image edge with `ski.segmentation.clear_border`,
   because they would distort the size measurements. Call the result `cleaned`.
3. Give each connected object its own integer with `ski.measure.label`. How many
   nuclei are there?
4. Show the labels on top of the image with `ski.color.label2rgb(labels,
   image=image, bg_label=0)`.

Note: older tutorials use `remove_small_objects(..., min_size=30)`. In
scikit-image 0.26 that argument is deprecated in favour of `max_size`, which
removes objects smaller than **or equal to** the value.

In [ ]:
# Your code here


## 5. Measure

`ski.measure.regionprops_table` measures every labelled object and returns a
dictionary that pandas can turn straight into a DataFrame.

1. Measure `label`, `area`, `eccentricity` and `intensity_mean` (pass
   `intensity_image=image` for the last one) into a DataFrame `nuclei`.
2. Plot a histogram of the areas. Is it one population, or is there a tail?
3. Look at the largest objects: are they big nuclei, or something else?
   (Hint: `nuclei.nlargest(5, "area")`, then find those labels in the image.)

In [ ]:
# Your code here


In [ ]:
# Your code here


In [ ]:
# Your code here


*Your answer here.*

## 6. Stretch

### Separate touching nuclei with a watershed

Touching nuclei form a single connected blob. The classic fix:

1. Compute the distance from each foreground pixel to the background:
   `sp.ndimage.distance_transform_edt(cleaned)`. Nucleus centres are maxima.
2. Find those maxima with `ski.feature.peak_local_max(distance, min_distance=7,
   labels=labels)`. It returns coordinates.
3. Turn them into a marker image (0 everywhere, 1, 2, 3, ... at the maxima).
4. Flood `-distance` from the markers with `ski.segmentation.watershed(-distance,
   markers, mask=cleaned)`.

How many nuclei do you count now? Compare `label2rgb` images before and after.

In [ ]:
# Your code here


### SPL exercises on the coins image

From [SPL, scikit-image](https://lectures.scientific-python.org/packages/scikit-image/index.html):

* **Segmentation:** load `ski.data.coins()`. Separate the coins from the background
  by trying several methods: Otsu thresholding, adaptive thresholding
  (`ski.filters.threshold_local`), and watershed or random walker segmentation.
  Post-process if needed.
* **Measurement:** from your binary coins image, label the coins and compute the
  size and eccentricity of each.
* **Colour:** open a colour image (for example `ski.data.astronaut()`), plot the
  histogram of each colour channel, then convert it to grey with
  `ski.color.rgb2gray` and plot that histogram.